In [5]:

# ============================================================
# NLP-BASED CLASSIFICATION AND INFORMATION EXTRACTION
# UNIVERSITY OF HYDERABAD ADMINISTRATIVE NOTICES
# CIA 1 - PHASE 1
#
# NOTE: No classifier is trained in Phase 1.
# ============================================================


# 1. IMPORT LIBRARIES
# ============================================================

from pathlib import Path
import re
import unicodedata
import warnings
import sys

import pandas as pd
import numpy as np
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import save_npz

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
print("Python:", sys.version)


# 2. DOWNLOAD / CHECK NLTK RESOURCES
# ============================================================

nltk_resources = [
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("corpora/stopwords", "stopwords"),
    ("corpora/wordnet", "wordnet"),
    ("corpora/omw-1.4", "omw-1.4")
]

for resource_path, package_name in nltk_resources:
    try:
        nltk.data.find(resource_path)
    except LookupError:
        print("Downloading:", package_name)
        nltk.download(package_name, quiet=True)

# Initialize NLP tools
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("NLTK resources initialized.")


# 3. FIND AND LOAD DATASET
# ============================================================

PROJECT_ROOT = Path.cwd()

CANDIDATE_NAMES = [
    "uoh_phase1_preprocessed_improved.csv",
    "uoh_phase1_preprocessed_final.csv",
    "uoh_phase1_dataset_with_ocr_candidates.csv",
    "uoh_admin_documents_cleaned_annotation_prep.csv"
]

matches = []

for name in CANDIDATE_NAMES:
    matches.extend(PROJECT_ROOT.rglob(name))

if not matches:
    raise FileNotFoundError(
        "Dataset not found. Check the data folder and project location."
    )

# Prefer data folder and improved dataset
matches.sort(
    key=lambda p: (
        0 if p.parent.name.lower() == "data" else 1,
        CANDIDATE_NAMES.index(p.name),
        len(p.parts)
    )
)

DATASET_PATH = matches[0]

print("\nSelected dataset:", DATASET_PATH)

df = pd.read_csv(DATASET_PATH)

print("\nDATASET INFORMATION")
print("-------------------")
print("Total records:", len(df))
print("Total columns:", len(df.columns))


# 4. SELECT TEXT COLUMN
# ============================================================

TEXT_CANDIDATES = [
    "original_text_for_nlp",
    "notice_text",
    "cleaned_text",
    "text",
    "processed_text"
]

TEXT_COLUMN = next(
    (col for col in TEXT_CANDIDATES if col in df.columns),
    None
)

if TEXT_COLUMN is None:
    raise ValueError("No suitable text column found in dataset.")

print("Selected text column:", TEXT_COLUMN)

# Preserve missing values as empty strings
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)

df["source_text"] = df[TEXT_COLUMN].str.strip()

print("Records with text:", (df["source_text"] != "").sum())
print("Records without text:", (df["source_text"] == "").sum())


# 5. TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# 6. TOKENIZATION, STOP-WORD REMOVAL,
#    STEMMING AND LEMMATIZATION
# ============================================================

def preprocess_text(text):

    normalized = normalize_text(text)

    if not normalized:
        return {
            "normalized_text": "",
            "tokens": [],
            "filtered_tokens": [],
            "stemmed_tokens": [],
            "lemmatized_tokens": [],
            "processed_text": ""
        }

    # Tokenization
    tokens = word_tokenize(normalized)

    # Stop-word removal
    filtered_tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    # Stemming
    stemmed_tokens = [
        stemmer.stem(word)
        for word in filtered_tokens
    ]

    # Lemmatization
    lemmatized_tokens = [
        lemmatizer.lemmatize(word)
        for word in filtered_tokens
    ]

    return {
        "normalized_text": normalized,
        "tokens": tokens,
        "filtered_tokens": filtered_tokens,
        "stemmed_tokens": stemmed_tokens,
        "lemmatized_tokens": lemmatized_tokens,
        "processed_text": " ".join(lemmatized_tokens)
    }


# 7. APPLY NLP PREPROCESSING
# ============================================================

print("\nStarting NLP preprocessing...")

results = df["source_text"].apply(preprocess_text)

df["normalized_text_new"] = results.apply(
    lambda x: x["normalized_text"]
)

df["tokens"] = results.apply(
    lambda x: x["tokens"]
)

df["filtered_tokens"] = results.apply(
    lambda x: x["filtered_tokens"]
)

df["stemmed_tokens"] = results.apply(
    lambda x: x["stemmed_tokens"]
)

df["lemmatized_tokens"] = results.apply(
    lambda x: x["lemmatized_tokens"]
)

df["processed_text_new"] = results.apply(
    lambda x: x["processed_text"]
)

print("Preprocessing completed.")

usable_documents = df["processed_text_new"].ne("").sum()

print("Documents processed:", usable_documents)
print("Documents without text:", len(df) - usable_documents)


# 8. DISPLAY PREPROCESSING EXAMPLE
# ============================================================

print("\nPREPROCESSING EXAMPLE")
print("---------------------")

example_index = df.index[df["source_text"].ne("")]

if len(example_index) > 0:

    sample = df.loc[example_index[0]]

    print("\nOriginal text:")
    print(sample["source_text"][:500])

    print("\nNormalized text:")
    print(sample["normalized_text_new"][:500])

    print("\nTokens:")
    print(sample["tokens"][:30])

    print("\nAfter stop-word removal:")
    print(sample["filtered_tokens"][:30])

    print("\nStemmed tokens:")
    print(sample["stemmed_tokens"][:30])

    print("\nLemmatized tokens:")
    print(sample["lemmatized_tokens"][:30])


# 9. REGULAR EXPRESSION EXTRACTION
# ============================================================

# Date pattern
DATE_PATTERN = (
    r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}"
    r"|\d{4}[/-]\d{1,2}[/-]\d{1,2}"
    r"|\d{1,2}\s+"
    r"(?:January|February|March|April|May|June|July|"
    r"August|September|October|November|December)"
    r"\s+\d{4})\b"
)

# Email pattern
EMAIL_PATTERN = (
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

# Labelled reference / notification number candidates
REFERENCE_PATTERN = (
    r"\b(?:Ref(?:erence)?\.?\s*(?:No\.?)?|"
    r"Memo\.?\s*(?:No\.?)?|"
    r"Notification\s*(?:No\.?)?|"
    r"Order\s*(?:No\.?)?|"
    r"File\s*(?:No\.?)?)"
    r"\s*[:#-]?\s*[A-Z0-9][A-Z0-9/._-]*"
)


def extract_regex(text):

    if not text:
        return {
            "dates": [],
            "emails": [],
            "reference_candidates": []
        }

    return {
        "dates": re.findall(
            DATE_PATTERN, text, flags=re.IGNORECASE
        ),

        "emails": re.findall(
            EMAIL_PATTERN, text
        ),

        "reference_candidates": re.findall(
            REFERENCE_PATTERN, text, flags=re.IGNORECASE
        )
    }


regex_results = df["source_text"].apply(extract_regex)

df["regex_dates"] = regex_results.apply(
    lambda x: x["dates"]
)

df["regex_emails"] = regex_results.apply(
    lambda x: x["emails"]
)

df["regex_reference_candidates"] = regex_results.apply(
    lambda x: x["reference_candidates"]
)


# 10. REGEX SUMMARY
# ============================================================

regex_summary = pd.DataFrame([
    {
        "field": "dates",
        "documents_with_candidates":
            df["regex_dates"].apply(bool).sum(),
        "total_candidates":
            df["regex_dates"].apply(len).sum()
    },
    {
        "field": "emails",
        "documents_with_candidates":
            df["regex_emails"].apply(bool).sum(),
        "total_candidates":
            df["regex_emails"].apply(len).sum()
    },
    {
        "field": "reference_candidates",
        "documents_with_candidates":
            df["regex_reference_candidates"].apply(bool).sum(),
        "total_candidates":
            df["regex_reference_candidates"].apply(len).sum()
    }
])

print("\nREGEX EXTRACTION SUMMARY")
print(regex_summary.to_string(index=False))


# 11. PREPARE DOCUMENTS FOR VECTORIZATION
# ============================================================

valid_mask = df["processed_text_new"].ne("")

documents = df.loc[
    valid_mask, "processed_text_new"
].tolist()

if len(documents) == 0:
    raise ValueError("No processed documents available for vectorization.")


# 12. BAG-OF-WORDS REPRESENTATION
# ============================================================

bow_vectorizer = CountVectorizer(
    max_features=3000,
    ngram_range=(1, 1)
)

bow_matrix = bow_vectorizer.fit_transform(documents)

print("\nBAG-OF-WORDS")
print("------------")
print("Matrix shape:", bow_matrix.shape)
print("Vocabulary size:", len(bow_vectorizer.vocabulary_))


# 13. TF-IDF REPRESENTATION
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 1)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("\nTF-IDF")
print("------")
print("Matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))


# 14. DISPLAY TOP VOCABULARY TERMS
# ============================================================

bow_vocabulary = pd.DataFrame({
    "term": bow_vectorizer.get_feature_names_out(),
    "total_frequency": np.asarray(
        bow_matrix.sum(axis=0)
    ).ravel()
}).sort_values(
    "total_frequency",
    ascending=False
)

tfidf_vocabulary = pd.DataFrame({
    "term": tfidf_vectorizer.get_feature_names_out(),
    "average_tfidf": np.asarray(
        tfidf_matrix.mean(axis=0)
    ).ravel()
}).sort_values(
    "average_tfidf",
    ascending=False
)

print("\nTOP 15 BAG-OF-WORDS TERMS")
print(bow_vocabulary.head(15).to_string(index=False))

print("\nTOP 15 TF-IDF TERMS")
print(tfidf_vocabulary.head(15).to_string(index=False))


# 15. DOCUMENT QUALITY CHECK
# ============================================================

quality_control = pd.DataFrame({
    "notice_id": (
        df["notice_id"]
        if "notice_id" in df.columns
        else df.index
    ),
    "has_original_text": df["source_text"].ne(""),
    "has_processed_text": df["processed_text_new"].ne(""),
    "original_character_count": df["source_text"].str.len(),
    "processed_character_count": df["processed_text_new"].str.len()
})


# 16. SAVE ALL OUTPUTS
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "cia1_phase1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Processed dataset
processed_csv = OUTPUT_DIR / "uoh_phase1_processed_dataset.csv"
df.to_csv(processed_csv, index=False)

# Sparse matrices
save_npz(
    OUTPUT_DIR / "uoh_bow_matrix.npz",
    bow_matrix
)

save_npz(
    OUTPUT_DIR / "uoh_tfidf_matrix.npz",
    tfidf_matrix
)

# Vocabulary files
bow_vocabulary.to_csv(
    OUTPUT_DIR / "bow_vocabulary.csv",
    index=False
)

tfidf_vocabulary.to_csv(
    OUTPUT_DIR / "tfidf_vocabulary.csv",
    index=False
)

# Regex summary
regex_summary.to_csv(
    OUTPUT_DIR / "regex_extraction_summary.csv",
    index=False
)

# Quality control
quality_control.to_csv(
    OUTPUT_DIR / "document_quality_control.csv",
    index=False
)


# 17. FINAL PROJECT SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("CIA 1 - PHASE 1 NLP PIPELINE COMPLETED")
print("=" * 60)

print("Input dataset:", DATASET_PATH.name)
print("Total records:", len(df))
print("Documents with usable text:", usable_documents)
print("Documents without usable text:", len(df) - usable_documents)
print("BoW matrix:", bow_matrix.shape)
print("TF-IDF matrix:", tfidf_matrix.shape)
print("Output directory:", OUTPUT_DIR.resolve())

print("\nGenerated files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)

print("\nClassifier training: NOT PERFORMED")
print("Classification is reserved for CIA 3 - Phase 2.")

print("\nPIPELINE EXECUTION SUCCESSFUL!")


Libraries imported successfully.
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Downloading: wordnet
Downloading: omw-1.4
NLTK resources initialized.

Selected dataset: C:\Users\Prit Sachani\Pictures\NLP_Projext\data\uoh_phase1_preprocessed_improved.csv

DATASET INFORMATION
-------------------
Total records: 500
Total columns: 39
Selected text column: original_text_for_nlp
Records with text: 455
Records without text: 45

Starting NLP preprocessing...
Preprocessing completed.
Documents processed: 455
Documents without text: 45

PREPROCESSING EXAMPLE
---------------------

Original text:
Page 1 of 2 School of Physics, University of Hyderabad, P.O. Central University, Prof. C.R. Rao Road, Gachibowli, Hyderabad- 500046 (India). Tel: 91-40-23134308; Fax: 91-40-23012800 School of Physics, University of Hyderabad Hyderabad – 500 046, INDIA Oct 14, 2026 NOTICE INVITING QUOTATION Ref. No. UoH/SoP/CARS-SDD-G /Prem/PUR/2026–2027/Equipment /Trolley mounted